# 1. What you'll learn

        - compare ridge, lasso, and elastic-net coefficient paths
- choose penalty strength without using test data
- see collinearity make unregularized coefficients unstable

        **The one question this notebook answers:** How do coefficient penalties trade a little bias for stability and feature selection?

# 2. Setup

This experiment uses correlated predictors and sparse known truth, the exact conditions that make the three penalties behave differently.

The seed is fixed at 0 for data, splits, optimization, and sampling.
Versions are printed so this execution remains an auditable experiment,
not a result detached from the software that produced it.

In [ ]:
from pathlib import Path
import platform
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet, RidgeCV, LassoCV, ElasticNetCV
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
sns.set_theme(style="whitegrid", palette="colorblind")

print(f"Python: {platform.python_version()}")
print(f"NumPy: {np.__version__} | pandas: {pd.__version__} | scikit-learn: {sklearn.__version__}")
print(f"Seed: {SEED}")

# 3. The data

We generate 600 rows with 12 correlated features. Only features 0, 3, and 7 truly affect the target with coefficients 3, −2, and 1.5. Correlation gives the model several nearly interchangeable explanations.

**Small example:** For coefficients [4,1,0,0,0], L1 size is 5 and L2 squared size is 17. Lasso charges 5λ and can zero entries; ridge charges 17λ and tends to shrink all smoothly.

In [ ]:
rng=np.random.default_rng(SEED);n=600;p=12
latent=rng.normal(size=(n,4));mix=rng.normal(size=(4,p));X=latent@mix+rng.normal(0,.35,(n,p))
true_beta=np.zeros(p);true_beta[[0,3,7]]=[3,-2,1.5];y=X@true_beta+rng.normal(0,1,n)
features=[f"x{i}" for i in range(p)];frame=pd.DataFrame(X,columns=features);frame["target"]=y
print("Shape:",frame.shape);print("Nonzero truth:",{features[i]:true_beta[i] for i in np.flatnonzero(true_beta)});print(frame.head(2).round(2))

# 4. Explore

We inspect correlation, marginal relationships, and target scale. A feature can look predictive only because it shadows a true feature, which is why coefficient stability matters.

The next code produces three purposeful panels. Read the written
takeaways underneath before moving on; the plots are evidence for later
modeling choices, not decoration.

In [ ]:
fig,axes=plt.subplots(1,3,figsize=(14,3.8))
sns.heatmap(frame[features].corr(),cmap="vlag",center=0,ax=axes[0],cbar=False);axes[0].set_title("Correlated predictors")
axes[1].scatter(frame.x0,frame.target,s=12,alpha=.35);axes[1].set(title="True feature x0",xlabel="x0",ylabel="target")
sns.histplot(frame.target,bins=25,kde=True,ax=axes[2],color="#176b66");axes[2].set_title("Target scale")
plt.tight_layout();plt.show()

**Takeaway:** Blocks of correlation mean several columns can substitute for one another.

**Takeaway:** A true effect is visible but still mixed with correlated contributions.

**Takeaway:** The continuous target has enough spread for RMSE differences to be meaningful.

# 5. Prepare

We split 60/20/20 and standardize from training only. Penalties act on coefficient size, so leaving one feature in dollars and another in fractions would make the penalty a unit convention rather than a modeling choice.

Any learned scaling, encoding, imputation, or resampling belongs after
the split—normally inside a pipeline. Doing it on all rows would let
held-out information influence training and would make evaluation too
optimistic.

In [ ]:
dev,test=train_test_split(frame,test_size=.2,random_state=SEED);train,val=train_test_split(dev,test_size=.25,random_state=SEED)
scaler=StandardScaler().fit(train[features]);Xtr=scaler.transform(train[features]);Xv=scaler.transform(val[features]);Xte=scaler.transform(test[features]);ytr=train.target.to_numpy();yv=val.target.to_numpy();yte=test.target.to_numpy()
alphas=np.logspace(-3,1.5,35);print("Shapes:",Xtr.shape,Xv.shape,Xte.shape);print("Alpha range:",alphas[[0,-1]])

# 6. Train

        We trace coefficients from weak to strong penalty. Ridge shares weight across correlated features; lasso selects a sparse representative; elastic net mixes both behaviors.

        Conceptually, fitting follows these steps:

        1. Standardize features so a unit coefficient has comparable meaning.
2. For each alpha, minimize squared error plus the chosen penalty.
3. Record every coefficient to form a path.
4. Use cross-validation inside training to select alpha for final candidates.

In [ ]:
paths={"ridge":[],"lasso":[],"elastic":[]}
for a in alphas:
    paths["ridge"].append(Ridge(alpha=a).fit(Xtr,ytr).coef_)
    paths["lasso"].append(Lasso(alpha=a,max_iter=5000,random_state=SEED).fit(Xtr,ytr).coef_)
    paths["elastic"].append(ElasticNet(alpha=a,l1_ratio=.5,max_iter=5000,random_state=SEED).fit(Xtr,ytr).coef_)
models={"ridge":RidgeCV(alphas=alphas).fit(Xtr,ytr),"lasso":LassoCV(alphas=alphas,cv=5,max_iter=5000,random_state=SEED).fit(Xtr,ytr),"elastic":ElasticNetCV(alphas=alphas,l1_ratio=[.2,.5,.8],cv=5,max_iter=5000,random_state=SEED).fit(Xtr,ytr)}
for name,m in models.items():print(name,"selected alpha",round(float(m.alpha_),4))

# 7. Evaluate

We compare test RMSE, number of active coefficients, and full paths. Sparsity is not automatically truth: correlated variables let lasso select one proxy while ridge keeps a stable group.

We report a baseline as well as the model and save the central figure as
`key_figure.png`. A model earns attention only by beating a simple,
honestly evaluated alternative.

In [ ]:
baseline=DummyRegressor(strategy="mean").fit(Xtr,ytr);rows=[]
for name,m in models.items():
    rows.append({"model":name,"test_rmse":mean_squared_error(yte,m.predict(Xte))**.5,"active":int(np.sum(np.abs(m.coef_)>1e-4))})
rows.append({"model":"mean baseline","test_rmse":mean_squared_error(yte,baseline.predict(Xte))**.5,"active":0});print(pd.DataFrame(rows).set_index("model").round(3))
fig,axes=plt.subplots(1,3,figsize=(14,4))
for ax,(name,values) in zip(axes,paths.items()):
    ax.plot(alphas,np.asarray(values));ax.set_xscale("log");ax.set(title=name,xlabel="alpha",ylabel="standardized coefficient")
plt.tight_layout();plt.savefig(Path.cwd()/"key_figure.png",dpi=160,bbox_inches="tight");plt.show()

**Takeaway:** All three penalties beat the mean; ridge keeps correlated groups smooth, lasso creates zeros, and elastic net offers a compromise rather than a universally superior answer.

# 8. Break it

We repeatedly resample a nearly duplicated pair and fit OLS versus ridge. Their combined effect is identifiable, but OLS arbitrarily moves weight between copies; ridge stabilizes the individual coefficients.

The following experiment deliberately violates an assumption or removes
a protection. Compare the printed damage with the healthy evaluation;
the failure is part of the lesson, not an accidental exception.

In [ ]:
rng=np.random.default_rng(SEED);base=rng.normal(size=250);Xa=np.column_stack([base,base+rng.normal(0,.01,250)]);ya=4*base+rng.normal(0,1,250)
coef_ols=[];coef_ridge=[]
for i in range(80):
    idx=rng.choice(len(ya),len(ya),replace=True);coef_ols.append(LinearRegression().fit(Xa[idx],ya[idx]).coef_);coef_ridge.append(Ridge(alpha=5).fit(Xa[idx],ya[idx]).coef_)
coef_ols=np.asarray(coef_ols);coef_ridge=np.asarray(coef_ridge)
print("OLS coefficient SD:",coef_ols.std(0).round(3));print("Ridge coefficient SD:",coef_ridge.std(0).round(3));print("OLS/Ridge instability ratio:",round(coef_ols.std()/coef_ridge.std(),1))

**Use this when…** many predictors are correlated, sample size is limited, or stable prediction matters more than an unpenalized fit.

        **Don't use this when…** you interpret a selected lasso feature as uniquely causal or skip scaling and let measurement units determine shrinkage.

        ## Try this

        1. Set l1_ratio to .1 and .9; watch elastic-net paths move toward ridge and lasso.
2. Add 50 pure-noise features and compare test RMSE with OLS.
3. Repeat the data with uncorrelated features; lasso should recover the sparse truth more cleanly.